# Café Tools and One OpenAI Tool Interaction

Tools allow a model to request operations that Python executes. A tool definition describes the operation and its inputs; it is separate from the function that retrieves the record. The returned observation is associated with the request before the model prepares its answer.

In this experiment you will retrieve sales, event and promotion records from three supplied café CSVs, then follow one event-tool interaction. Run every cell in order in a fresh kernel. Predict results first, inspect the returned evidence, change an input and explain what happened. The notebook is self-contained; no previous notebook must be run. Default offline mode uses labelled supplied model messages but performs the CSV lookups in Python. Optional live mode makes exactly two model requests per complete run; it does not run an agent loop.

Keep `sales.csv`, `campus_events.csv` and `promotions.csv` beside this notebook. Locally install the companion requirements (including the compatible Brotli decoder) and use a standard Python kernel. Browser Python supports the offline route. There is no key in this notebook.

## Load the Café Records

We will reuse three CSV files from Chapter 4. Keep them beside the notebook.

In [ ]:
import pandas as pd

sales = pd.read_csv("sales.csv")
events = pd.read_csv("campus_events.csv")
promotions = pd.read_csv("promotions.csv")


Dates use the format `YYYY-MM-DD`. Coffee has the product identifier `"P01"`.

These are synthetic teaching datasets. The Python functions will retrieve actual records from the supplied files, rather than return invented results.

## Build Three Lookup Tools

Each function selects matching rows and returns a list of dictionaries. An empty list means that no matching record was found.

### Retrieve Sales

In [ ]:
def get_sales(date, product_id):
    # Match both the date and the product.
    rows = sales[
        (sales["date"] == date)
        & (sales["product_id"] == product_id)
    ]

    columns = ["date", "product_name", "quantity_sold"]
    return rows[columns].to_dict("records")


### Retrieve Campus Events

In [ ]:
def get_events(date):
    # Retrieve events recorded on the requested date.
    rows = events[events["date"] == date]

    columns = ["date", "event_name", "expected_attendance"]
    return rows[columns].to_dict("records")


### Retrieve Product Promotions

In [ ]:
def get_promotions(date, product_id):
    # Exclude promotions for other products.
    rows = promotions[
        (promotions["date"] == date)
        & (promotions["product_id"] == product_id)
    ]

    columns = ["date", "product_id", "discount_percent"]
    return rows[columns].to_dict("records")


The functions perform different operations, but follow the same pattern: select the relevant records and return a small result.

## Python Experiment: Inspect the Records

Consider this investigation:

> How did coffee sales change between 10 and 11 February 2026, and what recorded information might help us investigate the difference?

**Predict:** Which information will each function return? Would finding an event or promotion establish why sales changed?

In [ ]:
print("Previous sales:")
print(get_sales("2026-02-10", "P01"))

print("\nSelected-day sales:")
print(get_sales("2026-02-11", "P01"))

print("\nCampus events:")
print(get_events("2026-02-11"))

print("\nCoffee promotions:")
print(get_promotions("2026-02-11", "P01"))


**Run:** Execute the lookups and examine the returned records.

**Change:** Try another date or product identifier.

**Observe and Explain:**

- What changed between the two sales records?
- Which contextual records might deserve further investigation?
- What information is still missing?

An empty sales result means that the requested sales record is unavailable; it does not establish zero sales. In the supplied event and promotion files, an empty result means that no matching event or promotion was recorded.

Expected event attendance is not a count of café customers. Similarly, a promotion occurring alongside a sales change does not establish that it caused the change.

## Describe a Tool to the Model

The model needs a description of the tool before it can request its use. A **tool definition** specifies the function’s name, purpose and expected inputs. The Python function remains responsible for performing the operation. This separation follows the OpenAI function-calling approach. [@openaiFunctionCalling]

We will begin with the event lookup:

In [ ]:
events_tool = {
    "type": "function",
    "name": "get_events",
    "description": (
        "Retrieve recorded campus events for a date. "
        "Attendance figures are expected attendance, "
        "not actual café visits."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "Date in YYYY-MM-DD format."
            }
        },
        "required": ["date"],
        "additionalProperties": False
    }
}


This definition tells the model how to request the lookup. Creating the dictionary does not execute `get_events()`.

The other two functions can be described in the same way. For this first live experiment, we will expose only the event tool so that the interaction remains easy to follow.

## Python Experiment: One Real Tool Interaction

We will ask:

> Was a campus event recorded on 11 February 2026? Report the event name and expected attendance, and explain what this information cannot establish about café sales.

The experiment has three parts:

1. The model requests the event lookup.
2. Python executes the function and returns its result.
3. The model answers using that result.

This requires **two OpenAI requests**. The Python lookup itself makes no external request.

**Predict:** What information should the model obtain from the tool? Which claims would go beyond the returned record?

### Prepare the OpenAI Client

Reuse the `.env` setup introduced in Section 2.3. Keep your own `OPENAI_API_KEY` and `OPENAI_MODEL` in the configuration file, outside the notebook.

In [ ]:
import json
import os
from pathlib import Path
from types import SimpleNamespace

# Offline is the default. Set True only when ready for two real API requests.
USE_OPENAI = False
client = None
model = "supplied-example"

if USE_OPENAI:
    import sys
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("Use offline mode in browser Python; live calls need a local Python kernel.")
    from dotenv import load_dotenv
    from openai import OpenAI

    # Section 2.3 setup extended only to locate the downloaded package root.
    config_folder = Path.cwd()
    for folder in [config_folder, *config_folder.parents]:
        if (folder / "COMPANION_VERSION").is_file():
            config_folder = folder
            break
    load_dotenv(config_folder / ".env")
    model = os.environ["OPENAI_MODEL"]
    client = OpenAI(timeout=45.0, max_retries=0)
    print("Live mode configured. Running the next request cell uses OpenAI.")
else:
    print("Offline mode: supplied messages, real Python CSV lookups, no API requests.")


The notebook retains the established configuration-file lookup so that downloaded packages find the `.env` at their root. Its default offline mode requires no key; switch to live mode explicitly before running the API cells.

Our familiar `get_completion()` helper returns text. Here, we use the Responses API directly because we need to inspect a tool request as well as the final text. The existing helper remains unchanged.

### Ask the Model to Request the Lookup

In [ ]:
messages = [
    {
        "role": "user",
        "content": (
            "Was a campus event recorded on 2026-02-11? "
            "Report its name and expected attendance. "
            "Explain what this information cannot establish "
            "about café sales."
        )
    }
]

if USE_OPENAI:
    response = client.responses.create(
        model=model,
        input=messages,
        tools=[events_tool],
        tool_choice={"type": "function", "name": "get_events"},
        max_output_tokens=300,
        store=False
    )
else:
    # A supplied stand-in for the model's request; not a live response.
    response = SimpleNamespace(output=[SimpleNamespace(
        type="function_call", name="get_events",
        arguments=json.dumps({"date": "2026-02-11"}),
        call_id="supplied-event-call"
    )])
    print("Supplied tool request: get_events(date='2026-02-11')")


For this controlled experiment, `tool_choice` requires the event lookup. The model supplies the date argument. In Section 5.3, we will let the model choose among several tools.

### Execute the Requested Function

The response contains the requested function name and its arguments. We read the arguments, execute the matching Python function and attach the result to the interaction.

In [ ]:
# Retain the model's response, including its tool request.
messages += response.output

for item in response.output:
    if item.type == "function_call" and item.name == "get_events":
        arguments = json.loads(item.arguments)

        # Python performs the lookup.
        observation = get_events(arguments["date"])

        print("Requested tool:", item.name)
        print("Arguments:", arguments)
        print("Observation:", observation)

        # Associate this result with the model's specific request.
        messages.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(observation)
        })


`json.loads()` converts the model’s JSON arguments into a Python dictionary. `json.dumps()` converts the returned records into text that can be sent back to the model.

The `call_id` connects the observation to the corresponding tool request.

### Obtain the Final Answer

Now return the observation to the model:

In [ ]:
if USE_OPENAI:
    # Second and final model request: no additional tools are offered.
    answer = client.responses.create(
        model=model,
        input=messages,
        max_output_tokens=300,
        store=False
    )
    print("LIVE MODEL ANSWER")
    print(answer.output_text)
    client.close()
else:
    print("SUPPLIED TEACHING ANSWER — NOT A MODEL OUTPUT")
    print("A Campus community gathering was recorded on 2026-02-11, with expected attendance of 200.")
    print("Expected attendance does not measure actual café customers or establish an effect on sales.")


No tools are supplied to this second request. Its purpose is to produce an answer from the information already returned.

**Observe and Explain:**

- Does the event name match the CSV record?
- Is the expected attendance reported correctly?
- Does the answer distinguish expected attendees from actual café customers?
- Does it avoid claiming that the event caused a sales increase?

A tool result provides evidence, but the final explanation still needs examination.





## Apply What You Have Learned

Change the question to another date while keeping the tool definition unchanged. Predict the lookup result before running the experiment.

If no event is recorded, check whether the answer communicates that absence without inventing an event or claiming that nothing happened on campus.

Without API access, run the three Python lookups directly and follow the notebook’s clearly labelled supplied tool interaction. This supports understanding of the execution steps, but does not produce fresh model decisions.

## Check Your Understanding

1. How does a tool definition differ from its Python implementation?
2. Which line actually executes the event lookup?
3. Why is the tool result associated with a `call_id`?
4. Why does this example make two model requests?
5. What would be needed to investigate the sales change beyond checking an event record?

**Key takeaway:** The model requests an operation, Python performs it, and the result is returned to the model. This section demonstrates one complete tool interaction. In **5.3**, we will build the program that manages successive requests across the sales, event and promotion tools.

## Your change and reflection

Call `get_events()` for a different date and inspect its records. If working offline, write a labelled reader-written answer from those records; the supplied request and report above remain fixed examples. If using live mode, change the date in `messages` and rerun from the client setup cell through the final answer. Each complete live run uses two additional requests. Do not rerun all cells with live mode enabled merely to display old results.

Explain the difference between missing sales and no recorded event. Identify who requested the operation, which line executed it and where the result was returned. A successful answer uses only the returned evidence. Full repeated tool selection is introduced in Section 5.3.